In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS asb.olist_gold
;

## Dimension tables

In [0]:
%sql
-- Geolocation (Deduplicated)
CREATE OR REPLACE TABLE asb.olist_gold.dim_geolocation AS
SELECT 
    geolocation_zip_code_prefix AS zip_code_prefix,
    MAX(geolocation_city) AS city,
    MAX(geolocation_state) AS state,
    AVG(CAST(geolocation_lat AS DOUBLE)) AS latitude,
    AVG(CAST(geolocation_lng AS DOUBLE)) AS longitude
FROM asb.olist_silver.valid_geolocation
GROUP BY geolocation_zip_code_prefix
;

In [0]:
%sql
-- Products
CREATE OR REPLACE TABLE asb.olist_gold.dim_products AS
SELECT 
    p.product_id,
    COALESCE(t.product_category_name_english, p.product_category_name, 'unknown') AS product_category,
    CAST(p.product_name_lenght AS INT) AS product_name_length,
    CAST(p.product_description_lenght AS INT) AS product_description_length,
    CAST(p.product_photos_qty AS INT) AS product_photos_quantity,
    CAST(p.product_weight_g AS DOUBLE) AS product_weight_grams,
    CAST(p.product_length_cm AS DOUBLE) AS product_length_cm,
    CAST(p.product_height_cm AS DOUBLE) AS product_height_cm,
    CAST(p.product_width_cm AS DOUBLE) AS product_width_cm
FROM asb.olist_silver.valid_products p
LEFT JOIN asb.olist_silver.valid_product_category_translation t 
    ON p.product_category_name = t.product_category_name
;

In [0]:
%sql
-- Sellers
CREATE OR REPLACE TABLE asb.olist_gold.dim_sellers AS
SELECT 
    s.seller_id,
    s.seller_zip_code_prefix AS zip_code_prefix,
    s.seller_city AS city,
    s.seller_state AS state,
    g.latitude,
    g.longitude
FROM asb.olist_silver.valid_sellers s
LEFT JOIN asb.olist_gold.dim_geolocation g 
    ON s.seller_zip_code_prefix = g.zip_code_prefix
;

In [0]:
%sql
-- Customers (with lifetime frequency profile)
CREATE OR REPLACE TABLE asb.olist_gold.dim_customers AS
WITH customer_frequency AS (
    SELECT 
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS lifetime_orders_count
    FROM asb.olist_silver.valid_customers c
    JOIN asb.olist_silver.valid_orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_zip_code_prefix AS zip_code_prefix,
    c.customer_city AS city,
    c.customer_state AS state,
    g.latitude,
    g.longitude,
    COALESCE(cf.lifetime_orders_count, 0) AS lifetime_orders_count,
    CASE WHEN COALESCE(cf.lifetime_orders_count, 0) > 1 THEN 'Repeat' ELSE 'One-Time' END AS customer_type
FROM asb.olist_silver.valid_customers c
LEFT JOIN asb.olist_gold.dim_geolocation g 
    ON c.customer_zip_code_prefix = g.zip_code_prefix
LEFT JOIN customer_frequency cf 
    ON c.customer_unique_id = cf.customer_unique_id
;

In [0]:
%sql
-- Date (calendar boundary 2016-01-01 to 2018-12-31)
CREATE OR REPLACE TABLE asb.olist_gold.dim_date AS
WITH date_range AS (
    SELECT EXPLODE(SEQUENCE(DATE '2016-01-01', DATE '2018-12-31', INTERVAL 1 DAY)) AS datum
)
SELECT 
    datum AS date_key,
    YEAR(datum) AS year,
    QUARTER(datum) AS quarter,
    CONCAT(YEAR(datum), '-Q', QUARTER(datum)) AS year_quarter,
    MONTH(datum) AS month,
    DATE_FORMAT(datum, 'yyyy-MM') AS year_month,
    DATE_FORMAT(datum, 'MMMM') AS month_name,
    DAYOFWEEK(datum) AS day_of_week,
    DATE_FORMAT(datum, 'EEEE') AS day_name,
    CASE WHEN DAYOFWEEK(datum) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend
FROM date_range
;

## Fact tables

In [0]:
%sql
-- Orders (Grain: 1 row per order)
CREATE OR REPLACE TABLE asb.olist_gold.fact_orders AS
SELECT 
    order_id,
    customer_id,
    order_status,
    CAST(order_purchase_timestamp AS TIMESTAMP) AS purchase_timestamp,
    CAST(CAST(order_purchase_timestamp AS TIMESTAMP) AS DATE) AS purchase_date_key,
    CAST(order_approved_at AS TIMESTAMP) AS approved_timestamp,
    CAST(order_delivered_carrier_date AS TIMESTAMP) AS carrier_delivered_timestamp,
    CAST(order_delivered_customer_date AS TIMESTAMP) AS customer_delivered_timestamp,
    CAST(order_estimated_delivery_date AS TIMESTAMP) AS estimated_delivery_timestamp,
    CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END AS is_delivered,
    -- Logistics Lead Time in days
    ROUND(
        (UNIX_TIMESTAMP(CAST(order_delivered_customer_date AS TIMESTAMP)) - 
         UNIX_TIMESTAMP(CAST(order_purchase_timestamp AS TIMESTAMP))) / 86400.0, 
        2
    ) AS delivery_duration_days,
    -- Estimated SLA Duration in days
    ROUND(
        (UNIX_TIMESTAMP(CAST(order_estimated_delivery_date AS TIMESTAMP)) - 
         UNIX_TIMESTAMP(CAST(order_purchase_timestamp AS TIMESTAMP))) / 86400.0, 
        2
    ) AS estimated_delivery_duration_days,
    -- SLA Breach Tag
    CASE 
        WHEN CAST(order_delivered_customer_date AS TIMESTAMP) > CAST(order_estimated_delivery_date AS TIMESTAMP) THEN 1 
        ELSE 0 
    END AS is_late_delivery
FROM asb.olist_silver.valid_orders
;

In [0]:
%sql
-- Order Items (Grain: 1 row per order item)
CREATE OR REPLACE TABLE asb.olist_gold.fact_order_items AS
SELECT 
    order_id,
    CAST(order_item_id AS INT) AS order_item_id,
    product_id,
    seller_id,
    CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_timestamp,
    CAST(price AS DECIMAL(10, 2)) AS price,
    CAST(freight_value AS DECIMAL(10, 2)) AS freight_value,
    (CAST(price AS DECIMAL(10, 2)) + CAST(freight_value AS DECIMAL(10, 2))) AS gross_item_revenue
FROM asb.olist_silver.valid_order_items
;

In [0]:
%sql
-- Order Payments (Grain: 1 row per payment transaction)
CREATE OR REPLACE TABLE asb.olist_gold.fact_order_payments AS
SELECT 
    order_id,
    CAST(payment_sequential AS INT) AS payment_sequential,
    payment_type,
    CAST(payment_installments AS INT) AS payment_installments,
    CAST(payment_value AS DECIMAL(10, 2)) AS payment_value
FROM asb.olist_silver.valid_order_payments
;

In [0]:
%sql
-- Order Reviews (Grain: 1 row per review response)
CREATE OR REPLACE TABLE asb.olist_gold.fact_order_reviews AS
SELECT 
    review_id,
    order_id,
    CAST(review_score AS INT) AS review_score,
    CAST(review_creation_date AS TIMESTAMP) AS review_creation_timestamp,
    CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp
FROM asb.olist_silver.valid_order_reviews
;

# Semantic layer

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS asb.semantic
;

In [0]:
%sql
-- View 1: Delivery SLA & Logistics Analytics
CREATE OR REPLACE VIEW asb.semantic.vw_delivery_logistics AS
SELECT 
    o.order_id,
    o.order_status,
    o.purchase_date_key,
    d.year,
    d.year_quarter,
    d.year_month,
    c.customer_id,
    c.city AS customer_city,
    c.state AS customer_state,
    c.customer_type,
    o.delivery_duration_days,
    o.estimated_delivery_duration_days,
    o.is_delivered,
    o.is_late_delivery
FROM asb.olist_gold.fact_orders o
LEFT JOIN asb.olist_gold.dim_customers c ON o.customer_id = c.customer_id
LEFT JOIN asb.olist_gold.dim_date d ON o.purchase_date_key = d.date_key
;

In [0]:
%sql
-- View 2: Sales, Margin & Product Performance
CREATE OR REPLACE VIEW asb.semantic.vw_order_item_sales AS
SELECT 
    i.order_id,
    i.order_item_id,
    o.purchase_date_key,
    d.year,
    d.year_month,
    i.product_id,
    p.product_category,
    i.seller_id,
    s.state AS seller_state,
    s.city AS seller_city,
    c.state AS customer_state,
    i.price AS item_price,
    i.freight_value,
    i.gross_item_revenue
FROM asb.olist_gold.fact_order_items i
JOIN asb.olist_gold.fact_orders o ON i.order_id = o.order_id
LEFT JOIN asb.olist_gold.dim_products p ON i.product_id = p.product_id
LEFT JOIN asb.olist_gold.dim_sellers s ON i.seller_id = s.seller_id
LEFT JOIN asb.olist_gold.dim_customers c ON o.customer_id = c.customer_id
LEFT JOIN asb.olist_gold.dim_date d ON o.purchase_date_key = d.date_key
;

In [0]:
%sql
-- View 3: Customer Satisfaction & Feedback Metrics
CREATE OR REPLACE VIEW asb.semantic.vw_order_satisfaction AS
SELECT 
    r.review_id,
    r.order_id,
    r.review_score,
    o.purchase_date_key,
    o.is_late_delivery,
    o.delivery_duration_days,
    c.state AS customer_state
FROM asb.olist_gold.fact_order_reviews r
JOIN asb.olist_gold.fact_orders o ON r.order_id = o.order_id
LEFT JOIN asb.olist_gold.dim_customers c ON o.customer_id = c.customer_id
;